# 07 — Modelos de Hugging Face para Forecasting

Tres enfoques con la librería `transformers` de HF:

| # | Modelo | Tipo | Ventaja |
|---|--------|------|---------|
| 1 | **TimeSeriesTransformer** | Fine-tunable | Arquitectura oficial HF para TS |
| 2 | **Chronos-T5-Small** (Amazon) | Zero-shot | No requiere entrenamiento |
| 3 | **Moirai** (Salesforce) | Zero-shot | Foundation model multi-variante |

### Instalación
```bash
pip install transformers accelerate
pip install git+https://github.com/amazon-science/chronos-forecasting.git
pip install uni2ts   # para Moirai
```


In [14]:
import sys, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
from utils import load_data, compute_rmse, make_submission, train_val_split, INDEX_COLS

data       = load_data()
train_full = data["train_indices"][INDEX_COLS]
test_dates = data["test_dates"].index
train, val = train_val_split(train_full, val_size=252)

N_STEPS_VAL  = len(val)         # horizonte de validacion
N_STEPS_TEST = len(test_dates)  # dinamico: 252, 262, etc.
print(f"Train: {train.shape}  |  Val: {val.shape}  |  Test: {N_STEPS_TEST}")


Train: (2095, 6)  |  Val: (252, 6)  |  Test: 262


---
## Modelo 1 — `TimeSeriesTransformer` (Hugging Face `transformers`)

Modelo probabilístico que usa distribuciones de student-t como salida.
Entrenamos un modelo por índice (o uno multi-output con listas de series).

Docs: https://huggingface.co/docs/transformers/model_doc/time_series_transformer


In [15]:
from transformers import (
    TimeSeriesTransformerConfig,
    TimeSeriesTransformerForPrediction,
)
import torch
from torch.utils.data import DataLoader, TensorDataset

LAGS_SEQ  = [1, 2, 3, 5, 7, 10, 20, 30, 60]
CONTEXT   = 64
PAST_LEN  = CONTEXT + max(LAGS_SEQ)  # longitud del tensor past_values
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print(f"CONTEXT={CONTEXT}, max_lag={max(LAGS_SEQ)}, PAST_LEN={PAST_LEN}")


Device: cuda
CONTEXT=64, max_lag=60, PAST_LEN=124


In [16]:
def build_hf_dataset(series: np.ndarray, past_len: int, horizon: int):
    """
    Construye tensores para TimeSeriesTransformer.
    past_values tiene shape (N, past_len) = (N, context + max_lag).
    past_time_features y past_observed_mask tienen la misma longitud past_len.
    """
    past_vals, fut_vals = [], []
    for i in range(past_len, len(series) - horizon + 1):
        past_vals.append(series[i - past_len : i])
        fut_vals.append(series[i : i + horizon])
    pv = torch.tensor(np.array(past_vals), dtype=torch.float32)
    fv = torch.tensor(np.array(fut_vals),  dtype=torch.float32)
    tp = torch.linspace(0, 1, past_len).unsqueeze(0).expand(len(pv), -1).unsqueeze(-1)
    tf = torch.linspace(0, 1, horizon).unsqueeze(0).expand(len(fv), -1).unsqueeze(-1)
    return pv, tp, fv, tf


In [17]:
from sklearn.preprocessing import StandardScaler

def train_hf_tst(train_series, col_name, horizon, epochs=30, batch=16, lr=1e-4):
    sc = StandardScaler()
    scaled = sc.fit_transform(train_series.reshape(-1, 1)).ravel()

    pv, tp, fv, tf = build_hf_dataset(scaled, PAST_LEN, horizon)
    obs_mask = torch.ones_like(pv)
    ds = TensorDataset(pv, tp, fv, tf, obs_mask)
    dl = DataLoader(ds, batch_size=batch, shuffle=True)

    cfg = TimeSeriesTransformerConfig(
        prediction_length=horizon, context_length=CONTEXT,
        lags_sequence=LAGS_SEQ, num_time_features=1,
        num_dynamic_real_features=0, num_static_categorical_features=0,
        num_static_real_features=0,
        d_model=64, encoder_layers=2, decoder_layers=2,
        encoder_attention_heads=4, decoder_attention_heads=4,
        encoder_ffn_dim=128, decoder_ffn_dim=128,
        dropout=0.1, distribution_output="student_t",
    )
    model = TimeSeriesTransformerForPrediction(cfg).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr)

    model.train()
    for ep in range(epochs):
        total = 0.0
        for pv_b, tp_b, fv_b, tf_b, om_b in dl:
            pv_b = pv_b.to(device); fv_b = fv_b.to(device)
            tp_b = tp_b.to(device); tf_b = tf_b.to(device)
            om_b = om_b.to(device)
            opt.zero_grad()
            out = model(past_values=pv_b, past_time_features=tp_b,
                        past_observed_mask=om_b, future_values=fv_b,
                        future_time_features=tf_b)
            out.loss.backward()
            opt.step()
            total += out.loss.item() * len(pv_b)
        if (ep + 1) % 10 == 0:
            print(f"  [{col_name}] Epoch {ep+1}/{epochs}  loss={total/len(ds):.5f}")

    model.eval()
    sc_last = torch.tensor(scaled[-PAST_LEN:], dtype=torch.float32).unsqueeze(0).to(device)
    tp_inf  = torch.linspace(0, 1, PAST_LEN).view(1, PAST_LEN, 1).to(device)
    tf_inf  = torch.linspace(0, 1, horizon).view(1, horizon, 1).to(device)
    om_inf  = torch.ones(1, PAST_LEN, dtype=torch.float32).to(device)
    with torch.no_grad():
        out = model.generate(past_values=sc_last, past_time_features=tp_inf,
                             past_observed_mask=om_inf, future_time_features=tf_inf)
    samples = out.sequences.cpu().numpy()
    median  = np.median(samples[0], axis=0)
    return sc.inverse_transform(median.reshape(-1, 1)).ravel(), model, sc

print("Funciones HF TimeSeriesTransformer definidas.")


Funciones HF TimeSeriesTransformer definidas.


In [5]:
tst_val_preds = {}
tst_models    = {}

print("Entrenando TimeSeriesTransformer por indice ...")
for col in INDEX_COLS:
    print(f"\n=== {col} ===")
    pred_arr, model, sc = train_hf_tst(train[col].values, col_name=col,
                                        horizon=N_STEPS_VAL, epochs=20)
    tst_val_preds[col] = pred_arr
    tst_models[col]    = (model, sc)

pred_tst_val = pd.DataFrame(tst_val_preds, index=val.index)
rmse_tst = compute_rmse(val, pred_tst_val)
print(f"\n[HF TimeSeriesTransformer] RMSE local = {rmse_tst:,.2f}")
per = np.sqrt(((val.values - pred_tst_val.values)**2).mean(axis=0))
for col, r in zip(INDEX_COLS, per): print(f"  {col}: {r:,.2f}")


Entrenando TimeSeriesTransformer por indice ...

=== Index_A ===
  [Index_A] Epoch 10/20  loss=-0.67135
  [Index_A] Epoch 20/20  loss=-1.02982

=== Index_B ===
  [Index_B] Epoch 10/20  loss=-0.67721
  [Index_B] Epoch 20/20  loss=-1.02039

=== Index_C ===
  [Index_C] Epoch 10/20  loss=-0.71734
  [Index_C] Epoch 20/20  loss=-0.99473

=== Index_D ===
  [Index_D] Epoch 10/20  loss=-0.68403
  [Index_D] Epoch 20/20  loss=-0.99875

=== Index_E ===
  [Index_E] Epoch 10/20  loss=-0.65456
  [Index_E] Epoch 20/20  loss=-0.97292

=== Index_F ===
  [Index_F] Epoch 10/20  loss=-0.67326
  [Index_F] Epoch 20/20  loss=-1.02187

[HF TimeSeriesTransformer] RMSE local = 1,808.30
  Index_A: 581.90
  Index_B: 439.36
  Index_C: 5.34
  Index_D: 1,496.07
  Index_E: 10.89
  Index_F: 8,316.26


In [6]:
tst_test_preds = {}
print("Reentrenando en datos completos (horizon=N_STEPS_TEST) ...")
for col in INDEX_COLS:
    print(f"  {col} ...", end=" ", flush=True)
    pred_arr, _, _ = train_hf_tst(train_full[col].values, col_name=col,
                                   horizon=N_STEPS_TEST, epochs=20)
    tst_test_preds[col] = pred_arr
    print("ok")

pred_tst_test = pd.DataFrame(tst_test_preds, index=test_dates)
make_submission(pred_tst_test, "submission_07a_hf_tst.csv")
pred_tst_test.head()


Reentrenando en datos completos (horizon=N_STEPS_TEST) ...
  Index_A ...   [Index_A] Epoch 10/20  loss=-0.69496
  [Index_A] Epoch 20/20  loss=-1.00069
ok
  Index_B ...   [Index_B] Epoch 10/20  loss=-0.69963
  [Index_B] Epoch 20/20  loss=-0.99031
ok
  Index_C ...   [Index_C] Epoch 10/20  loss=-0.60395
  [Index_C] Epoch 20/20  loss=-0.88257
ok
  Index_D ...   [Index_D] Epoch 10/20  loss=-0.65705
  [Index_D] Epoch 20/20  loss=-1.03192
ok
  Index_E ...   [Index_E] Epoch 10/20  loss=-0.64527
  [Index_E] Epoch 20/20  loss=-0.99591
ok
  Index_F ...   [Index_F] Epoch 10/20  loss=-0.59290
  [Index_F] Epoch 20/20  loss=-0.94768
ok
Submission saved: c:\Users\1jose\Desktop\previa hackatlon\hackathon\submissions\submission_07a_hf_tst.csv


,Index_A,Index_B,Index_C,Index_D,Index_E,Index_F
Date,,,,,,
2024-01-01,16570.150391,4607.912109,19.927599,16455.087891,128.273529,45622.164062
2024-01-02,16540.277344,4614.225586,19.915718,16377.217773,127.143005,45303.761719
2024-01-03,16281.227539,4567.702148,19.856247,16224.526367,125.921135,46525.378906
2024-01-04,16201.715820,4544.950195,20.004606,16172.123047,125.299316,46961.125000
2024-01-05,16219.648438,4514.916016,19.822762,16381.220703,125.688507,47748.128906


---
## Modelo 2 — Chronos-T5-Small (Amazon, zero-shot)

**Chronos** es un foundation model pre-entrenado sobre miles de datasets de series temporales.
No requiere fine-tuning: cargamos los pesos y predecimos directamente.

- Modelos disponibles: `amazon/chronos-t5-tiny`, `small`, `base`, `large`
- Paper: *Chronos: Learning the Language of Time Series* (Ansari et al. 2024)

```bash
pip install git+https://github.com/amazon-science/chronos-forecasting.git
```


In [7]:
def forecast_chronos(series_dict, forecast_dates, model_name="amazon/chronos-t5-small",
                     num_samples=20):
    try:
        from chronos import ChronosPipeline
    except ImportError:
        print("Chronos no instalado."); return None

    pipeline = ChronosPipeline.from_pretrained(model_name, device_map="cpu",
                                                torch_dtype=torch.float32)
    print(f"Modelo cargado: {model_name}")
    n_steps = len(forecast_dates)
    preds = {}
    for col, series in series_dict.items():
        context = torch.tensor(series.values, dtype=torch.float32)
        forecast = pipeline.predict(context=context, prediction_length=n_steps,
                                    num_samples=num_samples)
        preds[col] = np.median(forecast[0].numpy(), axis=0)
        print(f"  {col}: done")
    return pd.DataFrame(preds, index=forecast_dates)

print("Ejecutando Chronos en modo zero-shot (validacion) ...")
pred_chronos_val = forecast_chronos({col: train[col] for col in INDEX_COLS}, val.index)

if pred_chronos_val is not None:
    rmse_ch = compute_rmse(val, pred_chronos_val)
    print(f"\n[Chronos zero-shot] RMSE local = {rmse_ch:,.2f}")
    per = np.sqrt(((val.values - pred_chronos_val.values)**2).mean(axis=0))
    for col, r in zip(INDEX_COLS, per): print(f"  {col}: {r:,.2f}")


Ejecutando Chronos en modo zero-shot (validacion) ...
Chronos no instalado.


In [8]:
# Submission Chronos (datos completos)
if pred_chronos_val is not None:
    train_series_full = {col: train_full[col] for col in INDEX_COLS}
    pred_chronos_test = forecast_chronos(train_series_full, test_dates)
    if pred_chronos_test is not None:
        make_submission(pred_chronos_test, "submission_07b_chronos.csv")
        pred_chronos_test.head()


---
## Modelo 3 — Moirai (Salesforce, zero-shot)

**MOIRAI** (Unified Training of Universal Time Series Forecasting Transformers)
es otro foundation model de Salesforce, diseñado para series multivariantes.

- Modelos: `Salesforce/moirai-1.0-R-small`, `base`, `large`
- Paper: *Unified Training of Universal TS Forecasting Transformers* (Woo et al. 2024)

```bash
pip install uni2ts
```


In [9]:
def forecast_moirai(series_df, forecast_dates,
                    model_name="Salesforce/moirai-1.0-R-small", num_samples=20):
    try:
        from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
        from gluonts.dataset.pandas import PandasDataset
    except ImportError:
        print("Moirai/uni2ts no instalado."); return None

    n_steps = len(forecast_dates)
    dataset = PandasDataset(dict(series_df), freq="B")
    model = MoiraiForecast(
        module=MoiraiModule.from_pretrained(model_name),
        prediction_length=n_steps, context_length=200,
        patch_size="auto", num_samples=num_samples,
        target_dim=1, feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0,
    )
    predictor = model.create_predictor(batch_size=8)
    forecasts  = list(predictor.predict(dataset))
    preds = {}
    for fc, col in zip(forecasts, INDEX_COLS):
        preds[col] = np.median(fc.samples, axis=0)
        print(f"  {col}: done")
    return pd.DataFrame(preds, index=forecast_dates)

print("Ejecutando Moirai en modo zero-shot (validacion) ...")
pred_moirai_val = forecast_moirai(train, val.index)

if pred_moirai_val is not None:
    rmse_m = compute_rmse(val, pred_moirai_val)
    print(f"\n[Moirai zero-shot] RMSE local = {rmse_m:,.2f}")
    per = np.sqrt(((val.values - pred_moirai_val.values)**2).mean(axis=0))
    for col, r in zip(INDEX_COLS, per): print(f"  {col}: {r:,.2f}")


Ejecutando Moirai en modo zero-shot (validacion) ...
Moirai/uni2ts no instalado.


In [10]:
if pred_moirai_val is not None:
    pred_moirai_test = forecast_moirai(train_full, test_dates)
    if pred_moirai_test is not None:
        make_submission(pred_moirai_test, "submission_07c_moirai.csv")
        pred_moirai_test.head()


---
## Comparativa global y Mega-Ensemble

In [11]:
import os

all_results = {}

# Recoger RMSE de modelos disponibles
for tag, pred in [
    ("HF_TST",   pred_tst_val   if 'pred_tst_val'   in dir() else None),
    ("Chronos",  pred_chronos_val if 'pred_chronos_val' in dir() and pred_chronos_val is not None else None),
    ("Moirai",   pred_moirai_val  if 'pred_moirai_val'  in dir() and pred_moirai_val  is not None else None),
]:
    if pred is not None:
        all_results[tag] = compute_rmse(val, pred)

df_cmp = pd.DataFrame.from_dict(all_results, orient="index", columns=["RMSE_local"]).sort_values("RMSE_local")
print("\n=== Comparativa modelos HF ===")
print(df_cmp.round(2))



=== Comparativa modelos HF ===
        RMSE_local
HF_TST      1808.3


In [12]:
# Mega-ensemble: mezcla con todos los submissions disponibles
candidates = {}
for fname in os.listdir("submissions"):
    if fname.endswith(".csv"):
        df = pd.read_csv(f"submissions/{fname}", parse_dates=[0], index_col=0)
        candidates[fname] = df.reindex(val.index)[INDEX_COLS]

if candidates:
    stack  = np.stack([v.values for v in candidates.values()], axis=0)
    mega   = stack.mean(axis=0)
    pred_mega = pd.DataFrame(mega, index=val.index, columns=INDEX_COLS)
    rmse_mega = compute_rmse(val, pred_mega)
    print(f"\n[Mega-Ensemble ({len(candidates)} modelos)] RMSE local = {rmse_mega:,.2f}")
    print("Modelos incluidos:", list(candidates.keys()))



[Mega-Ensemble (5 modelos)] RMSE local = nan
Modelos incluidos: ['submission_01_baseline.csv', 'submission_02_arima.csv', 'submission_03_lgbm.csv', 'submission_05_ensemble.csv', 'submission_07a_hf_tst.csv']


In [13]:
# Submission mega-ensemble con datos de test
test_candidates = {}
for fname in os.listdir("submissions"):
    if fname.endswith(".csv"):
        df = pd.read_csv(f"submissions/{fname}", parse_dates=[0], index_col=0)
        df_reindexed = df.reindex(test_dates)[INDEX_COLS]
        if not df_reindexed.isnull().any().any():
            test_candidates[fname] = df_reindexed

if test_candidates:
    stack_test = np.stack([v.values for v in test_candidates.values()], axis=0)
    mega_test  = stack_test.mean(axis=0)
    pred_mega_test = pd.DataFrame(mega_test, index=test_dates, columns=INDEX_COLS)
    make_submission(pred_mega_test, "submission_08_mega_ensemble.csv")
    print(f"\nMega-ensemble guardado con {len(test_candidates)} modelos.")
    pred_mega_test.head()


Submission saved: c:\Users\1jose\Desktop\previa hackatlon\hackathon\submissions\submission_08_mega_ensemble.csv

Mega-ensemble guardado con 5 modelos.
